# # ML - 비지도학습 - Anomaly Detection 실습

[사용 제한 안내]
> 본 실습 코드는 교육 및 학습 목적으로만 제공됩니다.

[사전 준비 사항]
- 없음

[분석 요건]
- 분석 주제 : 머신러닝 기법을 활용하여 집값 이싱치 탐지 실습
- 분석 단위 : 주거 지역 (동네 수준의 생활권)
- 분석 변수 : 지역의 인구·주택 정보 (인구수, 가구수, 주택 연식, 방 개수 등)

[주요 실습 내용]
- Isolation Forest

# 데이타 준비
- 분석 데이타 : 캘리포니아 집값 데이터셋
- 데이타 설명 : 캘리포니아의 작은 지역(Census Block Group)별 인구·주택 정보를 이용하여 주택 가격를 예측하기 위함
- 변수 설명
| 컬럼명                | 한글명      | 의미                      | 비고               |
| ------------------ | -------- | ----------------------- | ---------------- |
| longitude          | 경도       | 해당 지역의 경도               | 연속형              |
| latitude           | 위도       | 해당 지역의 위도               | 연속형              |
| housing_median_age | 주택 중위 연식 | 해당 지역 주택의 중위 연식(년)      | 연속형              |
| total_rooms        | 전체 방 개수  | 해당 지역의 전체 방(Room) 수     | 연속형              |
| total_bedrooms     | 전체 침실 개수 | 해당 지역의 전체 침실(Bedroom) 수 | 연속형              |
| population         | 인구수      | 해당 지역의 총 인구수            | 연속형              |
| households         | 가구수      | 해당 지역의 총 가구 수           | 연속형              |
| median_income      | 중위소득     | 해당 지역 가구의 중위소득          | 연속형              |
| ocean_proximity    | 바다와의 거리  | 해당 지역의 바다와의 위치 관계       | **범주형**          |
| median_house_value | 주택 중위가격  | 해당 지역의 주택 중위가격(달러)      | **종속변수(Target)** |
- ocean_proximity 값 의미
| 값            | 의미           |
| ------------ | ------------ |
| `<1H OCEAN`  | 바다까지 1시간 이내  |
| `INLAND`     | 내륙 지역        |
| `NEAR OCEAN` | 바다 인접 지역     |
| `NEAR BAY`   | 만(Bay) 인접 지역 |
| `ISLAND`     | 섬 지역         |






데이타 불러오기

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(url)
df

컬럼정보 확인

In [ ]:
df.info()

[참고] info()에서 확인할 것:
  - 총 행/열 수
  - 각 컬럼의 데이터 타입
    - int : 연속형 (정수형)
    - float : 연속형 (실수형)
    - object : 보통 문자열 (범주형)
    - category : 범주형,  *범주 갯수가 적을 경우 메모리 효율화를 위해 object 타입 대신 사용, 그러나 object 을 사용해도 무방
    - datetime : 날짜형

분석 변수 선택

In [ ]:
cols = [
    "median_income",       # 중위소득
    "total_rooms",         # 전체 방 개수
    "population",          # 인구수
    "households",          # 가구수
    "median_house_value"   # 주택 중위가격
]

df = df[cols].copy()

# EDA

연속형 변수 기본 분포값 확인
 - 비즈니스관점에서 데이터 이상 여부 확인, 특히 음수(-), '0' 데이터 주의
 - Missing 건수 확인
 - 데이터 최소, 최대, 중심값 확인
 - 데이터 밀집도 확인

In [ ]:
import numpy as np

# 연속형 변수 기본 분포 확인
df.select_dtypes(include=[np.number]).describe()

# 데이타 변환

Missing 처리
- 연속형: 중앙값 대체

In [ ]:
# 연속형 변수 추출
num_cols = df.select_dtypes(include=[np.number]).columns

# 연속형 변수 중앙값 대체
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

Scaling
- Z-Score 변환

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

# Isolation Forest 분석
- One-Class Classification
- API 문서 : https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html

학습 및 예측

In [ ]:
from sklearn.ensemble import IsolationForest

# 모델 정의
iso = IsolationForest(
    n_estimators=100,
    contamination=0.05,   # 전체 데이터 중 약 5%를 이상치로 간주
    random_state=42
)

# 이상치 학습 및 예측
df["outlier"] = iso.fit_predict(df_scaled)

# 이상치 현황
outliers = df[df["outlier"] == -1]

print("전체 개수  :", len(df))
print("이상치 개수:", len(outliers))
print("이상치 비율:", len(outliers) / len(df) * 100)

# 정상 / 이상치 label 변수 생성, -1: 이상치, 1: 정상
df["outlier_label"] = df["outlier"].map({
    1: "Normal",
    -1: "Anomaly"
})
df

이상치 집단의 특성 분석
- EDA 분석

In [ ]:
# 이상치 / 정상 비교 요약
summary = df.groupby("outlier_label")[cols].mean()

summary.round(2)

- Decision Tree 분석

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# Target 생성 (Normal=0, Anomaly=1)
df["target"] = df["outlier_label"].map({
    "Normal": 0,
    "Anomaly": 1
})

# X, y 분리
X = df.drop(['outlier_label', 'target', 'outlier'], axis=1)
y = df['target']

# 모델 정의
model = DecisionTreeClassifier(
    max_depth=3,       # Tree 깊이
    random_state=42
)

# 모델 학습
model.fit(X, y)

# Tree 시각화 : 노드 건수 기준 출력
from sklearn.tree import export_graphviz
import graphviz

dot_data = export_graphviz(
    model,
    feature_names=X.columns,
    class_names=['Normal', 'Anomaly'],
    filled=True,
    rounded=True,
    proportion=False   # 건수 기준 출력
)
graphviz.Source(dot_data)

### [Self-Practice]

- 실습 조건
  - 분석 변수에 latitude, longitude, housing_median_age 3개를 추가해서 이상치 분석 수행, 총 8개 변수 : 기존 5개 + 신규 3개
  - 그 외 데이타 변환, 학습 조건 등은 동일 적용
- 실습 시나리오
   - 시나리오 1) 이상치 집단의 특성 분석에서 Tree 시각화 결과 Anomaly 비율이 가장 높은 집단과 가장 낮은 집단 (노드)의 규칙(Rule)을 각각 도출하세요.
   - 시나리오 2) 이상치 집단의 특성 분석에서 Normal / Anomaly 집단을 구분하는데 영향력이 큰 변수 순서는?  Random Forest 기준 Feature Importance 분석 활용